In [6]:
import pandas as pd
import os

supplementary_dir = "results/Supplementary_Files"

os.makedirs(supplementary_dir, exist_ok=True)

In [7]:
def build_supplementary_table(
    mwu_path,
    delong_path,
    variant_column,
    top_column,
    second_column,
    variant_names,
    pair_column,
):
    metric_names = {
        "P_Value": "MWU-pvalue",
        "Adjusted_P_Value": "MWU-adjusted-pvalue",
        "ROC_AUC": "ROC-AUC",
    }

    mwu_df = pd.read_csv(mwu_path, sep="\t")
    delong_df = pd.read_csv(delong_path, sep="\t")

    mwu_metrics = mwu_df.reindex(
        columns=["Dataset", "TF", variant_column, *metric_names]
    )
    ordered_columns = [
        (metric, variant) for variant in variant_names for metric in metric_names
    ]

    variant_values = (
        mwu_metrics[mwu_metrics[variant_column].isin(variant_names)]
        .set_index(["Dataset", "TF", variant_column])[list(metric_names)]
        .unstack(variant_column)
        .reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    )
    variant_values.columns = [
        f"{variant_names[variant]}_{metric_names[metric]}"
        for metric, variant in variant_values.columns
    ]

    delong_columns = [
        "Dataset",
        "TF",
        top_column,
        second_column,
        "DeLong_P_Value",
        "DeLong_P_Value_FDR_BH",
    ]
    summary = delong_df[delong_columns].copy()
    summary[pair_column] = (
        summary[top_column].map(variant_names)
        + ", "
        + summary[second_column].map(variant_names)
    )
    summary = summary.drop(columns=[top_column, second_column])

    result = (
        variant_values.reset_index()
        .merge(summary, on=["Dataset", "TF"], how="left")
        .rename(
            columns={
                "DeLong_P_Value": "DeLongs-pvalue",
                "DeLong_P_Value_FDR_BH": "DeLongs-adjusted-pvalue",
            }
        )
    )

    final_columns = (
        ["Dataset", "TF"]
        + list(variant_values.columns)
        + [
            pair_column,
            "DeLongs-pvalue",
            "DeLongs-adjusted-pvalue",
        ]
    )
    return result[final_columns]

## 1. Methods Supplementary File

In [8]:
method_names = {
    "z-aggregate": "z-aggregate",
    "viper": "viper",
    "ulm": "ulm",
    "zscore": "zscore",
}

methods_delongs_df = build_supplementary_table(
    mwu_path="results/Methods_MWU-Delongs/MWU_merged.tsv",
    delong_path="results/Methods_MWU-Delongs/DeLong_top2_merged.tsv",
    variant_column="Method",
    top_column="Top_Method",
    second_column="Second_Method",
    variant_names=method_names,
    pair_column="Top, Second",
)

methods_delongs_df.to_csv(
    f"{supplementary_dir}/Methods_Delongs_Comparison_Table.tsv", sep="\t", index=False
)
methods_delongs_df

,Dataset,TF,z-aggregate_MWU-pvalue,z-aggregate_MWU-adjusted-pvalue,z-aggregate_ROC-AUC,viper_MWU-pvalue,viper_MWU-adjusted-pvalue,viper_ROC-AUC,ulm_MWU-pvalue,ulm_MWU-adjusted-pvalue,ulm_ROC-AUC,zscore_MWU-pvalue,zscore_MWU-adjusted-pvalue,zscore_ROC-AUC,"Top, Second",DeLongs-pvalue,DeLongs-adjusted-pvalue
0,FrangiehIzar2021_RNA,CCND1,0.483509,0.863095,0.500478,0.808274,0.889101,0.489928,0.135018,0.270035,0.512746,0.066495,0.132991,0.517362,NaN,NaN,NaN
1,FrangiehIzar2021_RNA,CDKN1A,0.395770,0.863095,0.502392,0.213266,0.426533,0.507197,0.120642,0.265413,0.510606,0.044510,0.097922,0.515392,"zscore, ulm",0.000002,0.000013
2,FrangiehIzar2021_RNA,CDKN2A,0.632405,0.993780,0.496655,0.000730,0.002294,0.531478,0.000366,0.001151,0.533401,0.000043,0.000158,0.538837,"zscore, ulm",0.003797,0.010443
3,FrangiehIzar2021_RNA,DNMT1,0.890240,0.999738,0.480461,0.448481,0.657771,0.502061,0.440196,0.577921,0.502395,0.235551,0.345475,0.511469,NaN,NaN,NaN
4,FrangiehIzar2021_RNA,E2F1,0.999738,0.999738,0.458327,0.890574,0.932983,0.485223,0.949623,0.969018,0.480277,0.872095,0.913624,0.486344,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,WesselsSatija2023,HDAC3,0.993613,0.993613,0.417969,0.002125,0.005667,0.594232,0.000405,0.003241,0.610391,0.004666,0.012443,0.585686,"ulm, viper",0.311268,0.516758
665,WesselsSatija2023,MYB,0.005282,0.010564,0.708166,0.144952,0.193270,0.586219,0.093726,0.149962,0.607364,0.131611,0.200419,0.591141,"z-aggregate, ulm",0.135269,0.338171
666,WesselsSatija2023,SSRP1,0.200903,0.292318,0.545100,0.200237,0.228842,0.545253,0.175366,0.200419,0.550222,0.175366,0.200419,0.550222,NaN,NaN,NaN
667,WesselsSatija2023,SUPT16H,0.000312,0.002463,0.598658,0.001132,0.004530,0.588238,0.000908,0.003631,0.590138,0.000908,0.007263,0.590138,"z-aggregate, ulm",0.128115,0.338171


## 2. Weights Supplementary File

In [9]:
weight_names = {
    "CORRELATION": "Correlation",
    "NONZERORATE": "Nonzero-Rate",
    "SPECIFICITY": "Specificity",
    "UNIFORM": "Uniform",
}

weights_delongs_df = build_supplementary_table(
    mwu_path="results/Weights_MWU-Delongs/MWU_merged.tsv",
    delong_path="results/Weights_MWU-Delongs/DeLong_top2_merged.tsv",
    variant_column="Weight",
    top_column="Top_Weight",
    second_column="Second_Weight",
    variant_names=weight_names,
    pair_column="Top, Second",
)

weights_delongs_df.to_csv(
    f"{supplementary_dir}/Weights_Delongs_Comparison_Table.tsv", sep="\t", index=False
)
weights_delongs_df

,Dataset,TF,Correlation_MWU-pvalue,Correlation_MWU-adjusted-pvalue,Correlation_ROC-AUC,Nonzero-Rate_MWU-pvalue,Nonzero-Rate_MWU-adjusted-pvalue,Nonzero-Rate_ROC-AUC,Specificity_MWU-pvalue,Specificity_MWU-adjusted-pvalue,Specificity_ROC-AUC,Uniform_MWU-pvalue,Uniform_MWU-adjusted-pvalue,Uniform_ROC-AUC,"Top, Second",DeLongs-pvalue,DeLongs-adjusted-pvalue
0,FrangiehIzar2021_RNA,CCND1,0.868610,0.909973,0.487059,0.397406,0.794812,0.503005,0.777160,0.986392,0.491187,0.483509,0.863095,0.500478,NaN,NaN,NaN
1,FrangiehIzar2021_RNA,CDKN1A,0.295440,0.722187,0.504865,0.451176,0.827156,0.501110,0.594600,0.986392,0.497833,0.395770,0.863095,0.502392,NaN,NaN,NaN
2,FrangiehIzar2021_RNA,CDKN2A,0.825418,0.909973,0.490740,0.725301,0.999978,0.494079,0.946318,0.986392,0.484075,0.632405,0.993780,0.496655,NaN,NaN,NaN
3,FrangiehIzar2021_RNA,DNMT1,0.348779,0.767313,0.506184,0.725656,0.999978,0.490456,0.946991,0.986392,0.474278,0.890240,0.999738,0.480461,NaN,NaN,NaN
4,FrangiehIzar2021_RNA,E2F1,0.998515,0.998515,0.464298,0.999978,0.999978,0.450931,0.976382,0.986392,0.476155,0.999738,0.999738,0.458327,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664,WesselsSatija2023,HDAC3,0.003269,0.008716,0.589619,0.976045,0.976045,0.434834,0.970633,0.970633,0.437733,0.993613,0.993613,0.417969,"Correlation, Specificity",0.012308,0.036925
665,WesselsSatija2023,MYB,0.009888,0.019776,0.689756,0.003681,0.007363,0.718192,0.004461,0.011896,0.712906,0.005282,0.010564,0.708166,"Nonzero-Rate, Specificity",0.933484,0.971238
666,WesselsSatija2023,SSRP1,0.098409,0.131212,0.569408,0.343924,0.458565,0.521633,0.321196,0.428262,0.524996,0.200903,0.292318,0.545100,NaN,NaN,NaN
667,WesselsSatija2023,SUPT16H,0.001321,0.008716,0.586719,0.000327,0.001309,0.598296,0.000466,0.003730,0.595474,0.000312,0.002463,0.598658,"Uniform, Nonzero-Rate",0.971238,0.971238


## 3. Priors Supplementary Files

Separate tables are generated for the AllAvailableTFs and CommonTFsOnly analyses. Each table includes a Best3 block from the No-Ensemble DeLong comparison and a Best4 block from the With-Ensemble DeLong comparison.

In [10]:
prior_names = {
    "causalpath": "CausalPath",
    "collectri": "CollecTRI",
    "dorothea": "DoRothEA",
    "ensemble": "Ensemble",
}

def build_priors_supplementary_table(
    mwu_path,
    best3_delong_path,
    best4_delong_path,
):
    metric_names = {
        "P_Value": "MWU-pvalue",
        "Adjusted_P_Value": "MWU-adjusted-pvalue",
        "ROC_AUC": "ROC-AUC",
    }

    mwu_df = pd.read_csv(mwu_path, sep="\t")
    mwu_metrics = mwu_df.reindex(
        columns=["Dataset", "TF", "Prior", *metric_names]
    )
    ordered_columns = [
        (metric, prior) for prior in prior_names for metric in metric_names
    ]
    prior_values = (
        mwu_metrics[mwu_metrics["Prior"].isin(prior_names)]
        .set_index(["Dataset", "TF", "Prior"])[list(metric_names)]
        .unstack("Prior")
        .reindex(columns=pd.MultiIndex.from_tuples(ordered_columns))
    )
    prior_values.columns = [
        f"{prior_names[prior]}_{metric_names[metric]}"
        for metric, prior in prior_values.columns
    ]

    def build_delong_block(delong_path, block_name):
        delong_df = pd.read_csv(delong_path, sep="\t")
        summary = delong_df[
            [
                "Dataset",
                "TF",
                "Top_Prior",
                "Second_Prior",
                "DeLong_P_Value",
                "DeLong_P_Value_FDR_BH",
            ]
        ].copy()
        top_prior = summary["Top_Prior"].map(prior_names)
        second_prior = summary["Second_Prior"].map(prior_names)
        pair_column = f"{block_name}_Top, Second"
        summary[pair_column] = top_prior.where(
            second_prior.isna(), top_prior + ", " + second_prior
        )
        return summary.drop(columns=["Top_Prior", "Second_Prior"]).rename(
            columns={
                "DeLong_P_Value": f"{block_name}_DeLongs-pvalue",
                "DeLong_P_Value_FDR_BH": f"{block_name}_DeLongs-adjusted-pvalue",
            }
        )

    best3_summary = build_delong_block(best3_delong_path, "Best3")
    best4_summary = build_delong_block(best4_delong_path, "Best4")
    result = (
        prior_values.reset_index()
        .merge(best3_summary, on=["Dataset", "TF"], how="left")
        .merge(best4_summary, on=["Dataset", "TF"], how="left")
    )

    return result[
        ["Dataset", "TF"]
        + list(prior_values.columns)
        + [
            "Best3_Top, Second",
            "Best3_DeLongs-pvalue",
            "Best3_DeLongs-adjusted-pvalue",
            "Best4_Top, Second",
            "Best4_DeLongs-pvalue",
            "Best4_DeLongs-adjusted-pvalue",
        ]
    ]

prior_supplementary_configs = {
    "AllAvailableTFs": {
        "mwu_filename": "MWU_merged_AllAvailableTFs.tsv",
        "best3_delong_filename": "DeLong_top2_No-Ensemble_AllAvailableTFs.tsv",
        "best4_delong_filename": "DeLong_top2_With-Ensemble_AllAvailableTFs.tsv",
    },
    "CommonTFsOnly": {
        "mwu_filename": "MWU_merged_CommonTFsOnly.tsv",
        "best3_delong_filename": "DeLong_top2_No-Ensemble_CommonTFsOnly.tsv",
        "best4_delong_filename": "DeLong_top2_With-Ensemble_CommonTFsOnly.tsv",
    },
}

prior_table_summary = []
for tf_set, filenames in prior_supplementary_configs.items():
    priors_delongs_df = build_priors_supplementary_table(
        mwu_path=f"results/Priors_MWU-Delongs/{filenames['mwu_filename']}",
        best3_delong_path=f"results/Priors_MWU-Delongs/{filenames['best3_delong_filename']}",
        best4_delong_path=f"results/Priors_MWU-Delongs/{filenames['best4_delong_filename']}",
    )
    output_path = f"{supplementary_dir}/Priors_Delongs_Comparison_Table_{tf_set}.tsv"
    priors_delongs_df.to_csv(output_path, sep="\t", index=False)
    prior_table_summary.append({"TF set": tf_set, "Rows": len(priors_delongs_df), "File": output_path})

pd.DataFrame(prior_table_summary)

,TF set,Rows,File
0,AllAvailableTFs,1300,results/Supplementary_Files/Priors_Delongs_Com...
1,CommonTFsOnly,225,results/Supplementary_Files/Priors_Delongs_Com...
